In [1]:
# =========================================
# BLOCK STACKING - FULL BIO PRECISION MODEL
# =========================================

import cv2
import numpy as np
import mediapipe as mp
from scipy.signal import find_peaks
import math

In [2]:
# MediaPipe Setup (Hands + Pose)

mpHands = mp.solutions.hands
hands = mpHands.Hands(min_detection_confidence=0.6)

mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence=0.6)

mpDraw = mp.solutions.drawing_utils

DRAWING_SPEC_LANDMARK = mpDraw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2)
DRAWING_SPEC_CONNECTION = mpDraw.DrawingSpec(color=(0,0,0), thickness=2)

In [3]:
# Utility Functions

def safe_mean(arr, default=0.0):
    return sum(arr)/len(arr) if len(arr) > 0 else default

def safe_std(arr, default=0.0):
    return float(np.std(arr)) if len(arr) > 0 else default

In [4]:
def dist(a,b):
    return np.sqrt((a.x-b.x)**2 + (a.y-b.y)**2)

In [5]:
def compute_hand_quality(hand_data):
    """
    hand_data = {
        'thumb': [(x,y), ...],
        'index': [(x,y), ...],
        'middle': [(x,y), ...],
        'ring': [(x,y), ...],
        'pinky': [(x,y), ...],
        'wrist': [(x,y), ...]
    }
    """

    # -------------------------------
    # Convert arrays
    # -------------------------------
    thumb = np.array(hand_data['thumb'])
    index = np.array(hand_data['index'])
    middle = np.array(hand_data['middle'])
    ring = np.array(hand_data['ring'])
    pinky = np.array(hand_data['pinky'])
    wrist = np.array(hand_data['wrist'])

    n = min(len(thumb), len(index), len(middle), len(ring), len(pinky))
    if n < 5:
        return 1

    thumb = thumb[:n]
    index = index[:n]
    middle = middle[:n]
    ring = ring[:n]
    pinky = pinky[:n]

    # -------------------------------
    # 1. PINCER PRECISION
    # -------------------------------
    pinch_dist = np.linalg.norm(thumb - index, axis=1)
    pinch_score = np.mean(pinch_dist)

    # -------------------------------
    # 2. GRIP TYPE (multi-finger usage)
    # -------------------------------
    grip_complexity = np.mean([
        np.linalg.norm(thumb - middle, axis=1),
        np.linalg.norm(thumb - ring, axis=1),
        np.linalg.norm(thumb - pinky, axis=1)
    ])

    # -------------------------------
    # 3. FINGER INDEPENDENCE
    # (variance between fingers)
    # -------------------------------
    finger_spread = np.std([
        index[:,1],
        middle[:,1],
        ring[:,1],
        pinky[:,1]
    ])

    # -------------------------------
    # 4. SMOOTHNESS (JERK)
    # -------------------------------
    vel = np.diff(index[:,1])
    accel = np.diff(vel) if len(vel) > 1 else np.array([0])
    jerk = np.diff(accel) if len(accel) > 1 else np.array([0])
    jerk_val = np.std(jerk)

    # -------------------------------
    # 5. STABILITY (TREMOR)
    # -------------------------------
    stability = np.std(index[:,0]) + np.std(index[:,1])

    # -------------------------------
    # 6. MOVEMENT EFFICIENCY
    # -------------------------------
    path = np.sum(np.linalg.norm(np.diff(index, axis=0), axis=1))
    displacement = np.linalg.norm(index[-1] - index[0]) + 1e-6
    efficiency = displacement / path

    # =========================================================
    # NORMALIZED SCORING (0–1 → weighted)
    # =========================================================

    score = 0

    # Precision (smaller is better)
    if pinch_score < 0.03: score += 2
    elif pinch_score < 0.06: score += 1

    # Grip maturity
    if grip_complexity < 0.1: score += 1

    # Independence
    if finger_spread > 0.01: score += 1

    # Smoothness
    if jerk_val < 0.005: score += 2
    elif jerk_val < 0.01: score += 1

    # Stability
    if stability < 0.01: score += 1

    # Efficiency
    if efficiency > 0.7: score += 1

    # -------------------------------
    # FINAL QUALITY SCORE (1–5)
    # -------------------------------
    quality_score = max(1, min(5, round(score)))

    return quality_score

In [8]:
def final_block_score(blocks, quality, age_group):

    # -------------------------------
    # AGE NORMS (BLOCKS)
    # -------------------------------
    norms = {
        "2.5-3": (3, 4),
        "3-4": (5, 6),
        "4-5": (7, 8)
    }

    low, high = norms.get(age_group, (5,6))

    # -------------------------------
    # FLOOR CONDITION
    # -------------------------------
    if blocks <= 1 or quality < 2:
        return 1

    # -------------------------------
    # SCORE 5 (EXCELLENT)
    # >= above norm + high quality
    # -------------------------------
    if blocks >= high + 1 and quality >= 4.5:
        return 5

    # -------------------------------
    # SCORE 4 (ABOVE AVG)
    # slightly below + good quality
    # -------------------------------
    if blocks >= high and 4.0 <= quality < 4.5:
        return 4

    # -------------------------------
    # SCORE 3 (AT NORM)
    # -------------------------------
    if blocks >= low and 3.0 <= quality < 4.0:
        return 3

    # -------------------------------
    # SCORE 2 (BELOW EXPECTATION)
    # -------------------------------
    if blocks >= low - 2 and 2.0 <= quality < 3.0:
        return 2

    # -------------------------------
    # DEFAULT
    # -------------------------------
    return 1

In [9]:
def block_stacking_precision(path, age_group='3-4'):

    cap = cv2.VideoCapture(path)

    # -------- STORAGE --------
    hand_data = {
        'thumb': [],
        'index': [],
        'middle': [],
        'ring': [],
        'pinky': [],
        'wrist': []
    }
    velocities, accelerations, jerks = [], [], []
    left_y, right_y = [], []

    drops = 0
    blocks = 0

    prev_y, prev_vel, prev_acc = None, None, None

    while True:
        ret, frame = cap.read()
        if not ret: break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        if results.multi_hand_landmarks:

            for hand_landmarks in results.multi_hand_landmarks:

                mpDraw.draw_landmarks(frame, hand_landmarks, mpHands.HAND_CONNECTIONS)

                lm = hand_landmarks.landmark

                lm = hand_landmarks.landmark

                # -------- GRIP --------
                hand_data['thumb'].append((lm[4].x, lm[4].y))
                hand_data['index'].append((lm[8].x, lm[8].y))
                hand_data['middle'].append((lm[12].x, lm[12].y))
                hand_data['ring'].append((lm[16].x, lm[16].y))
                hand_data['pinky'].append((lm[20].x, lm[20].y))
                hand_data['wrist'].append((lm[0].x, lm[0].y))

                # # -------- GRIP --------
                # thumb_index.append(dist(lm[4], lm[8]))
                # thumb_middle.append(dist(lm[4], lm[12]))

                # -------- POSITION --------
                y = lm[8].y

                if prev_y is not None:
                    vel = y - prev_y
        
                # velocity
                if prev_y is not None:
                    vel = y - prev_y
                    velocities.append(vel)

                    # acceleration
                    if prev_vel is not None:
                        acc = vel - prev_vel
                        accelerations.append(acc)

                        # jerk
                        if prev_acc is not None:
                            jerk = acc - prev_acc
                            jerks.append(jerk)

                        prev_acc = acc
                    prev_vel = vel

                    # detect placement
                    if vel > 0.02: blocks += 1
                    if vel > 0.05: drops += 1

                prev_y = y

                # bilateral
                if lm[8].x < 0.5:
                    left_y.append(y)
                else:
                    right_y.append(y)

        cv2.imshow("Block Precision", frame)
        if cv2.waitKey(1) & 0xFF == 27: break

    cap.release()
    cv2.destroyAllWindows()

    # =========================
    # NORMALIZED QUALITY SCORES
    quality = compute_hand_quality(hand_data)

    # final score
    final = final_block_score(blocks, quality, age_group)

    # =========================
    # OUTPUT
    print("------ BLOCK STACKING RESULT ------")
    print(f"Blocks: {blocks}")
    print(f"Drops: {drops}")
    print(f"Quality Score: {quality}")
    print(f"Final Score: {final}")

    return final, quality

In [10]:
path = "data/block_activity.mp4"
print(block_stacking_precision(path))

------ BLOCK STACKING RESULT ------
Blocks: 88
Drops: 24
Quality Score: 3
Final Score: 3
(3, 3)
